# Predicting energikarakter for Norwegian homes

This notebook builds a model that predicts the **energikarakter** (energy grade A–G) of a home
from a few features you could know about your own residence: building type, year built,
construction material and location.

**Data:** Enova's register of energy certificates (*energiattester*), published as open data
under [NLOD 2.0](https://data.norge.no/nlod/no/2.0) via the [Enova data portal](https://data.enova.no/).

**Steps:**
1. Load the data
2. Inspect it (shape, types, missing values)
3. Clean it (homes only, valid values, one certificate per dwelling)
4. Explore it (EDA)
5. Feature engineering
6. Train/test split and preprocessing pipeline
7. Train and compare models with cross-validation
8. Tune the best model
9. Evaluate on the test set
10. Interpret the model (which features matter)
11. Save the model and use it for predictions

### Why regression and not a CNN?
The energy grade is **ordinal**: A < B < … < G. We encode it as a number (A=1 … G=7), run regression,
and round the result to the nearest grade. Unlike classification, this means predicting "C" when the
answer is "B" counts as a smaller error than predicting "G".

A CNN (convolutional neural network) is built for data with spatial structure, like images. Our data is
**tabular** (one row per dwelling, a few columns), and on tabular data tree-based models such as
gradient boosting usually beat neural networks. They are also faster and easier to interpret, so we
stick with scikit-learn.

## 0. Setup

In [12]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split, cross_validate, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

load_dotenv()  # reads ENOVA_API_KEY from .env

RANDOM_STATE = 42
ROOT = Path("..")  # the notebook lives in notebooks/, data and models in the project root
DATA_DIR = Path(os.environ.get("ENERGI_DATA_DIR", ROOT / "data"))
MODEL_DIR = ROOT / "models"

# The full register has over a million rows. Set e.g. 200_000 for faster experimentation, None for everything.
SAMPLE_SIZE = None

## 1. Load the data

Enova's API requires a free subscription key:
1. Create a user at https://data.enova.no/ and subscribe to the **"Energimerkesystemet - Offentlige data"** API (v1).
2. Put the key in a `.env` file in the project folder as `ENOVA_API_KEY=<key>` (or download the CSV files by hand
   from the portal and put them in `data/`).

We use **v1**, which has the certificates up to and including 2025. **v2** only covers certificates
from 2026 onwards, issued under the new regulation (*energimerkeforskriften* of 1 January 2026), where
the grade is calculated differently. Mixing the two would give the same letter different meanings,
so we leave v2 out.

The cell below downloads one file per year into `data/` if they don't already exist, then reads
all the CSV files in the folder.

In [18]:
import requests

API_URL = "https://api.data.enova.no/ems/offentlige-data/v1/Fil/{year}"
YEARS = range(2010, 2026)  # v1 only has files up to and including 2025


def download_year(year: int, api_key: str) -> Path | None:
    target = DATA_DIR / f"energiattester_{year}.csv"
    if target.exists():
        return target
    r = requests.get(API_URL.format(year=year), headers={"x-api-key": api_key}, timeout=120)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    # The API responds with a link to the CSV file (as plain text or JSON), not the file itself
    link = r.text.strip()
    if link.startswith("{"):
        link = next(v for v in r.json().values() if isinstance(v, str) and v.startswith("http"))
    link = link.strip('"')
    if link.startswith("http"):
        r = requests.get(link, timeout=600)
        r.raise_for_status()
    target.write_bytes(r.content)
    return target


DATA_DIR.mkdir(exist_ok=True)
api_key = os.environ.get("ENOVA_API_KEY")
if api_key:
    for year in YEARS:
        path = download_year(year, api_key)
        print(year, "->", path)

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files in {DATA_DIR.resolve()}. Set ENOVA_API_KEY or download the files by hand.")
csv_files

FileNotFoundError: No CSV files in C:\Users\Henrik\Projects\energikarakter\data. Set ENOVA_API_KEY or download the files by hand.

In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    # Figure out whether the file uses ; or , as separator
    with open(path, encoding="utf-8-sig") as f:
        header = f.readline()
    sep = ";" if header.count(";") > header.count(",") else ","
    return pd.read_csv(path, sep=sep, encoding="utf-8-sig", low_memory=False)


raw = pd.concat([read_csv(p) for p in csv_files], ignore_index=True)
print(raw.shape)
raw.head()

### Column names
Enova's files use the column names below. If a name doesn't match your file (Enova sometimes
changes the format), fix it here; everything else in the notebook uses these variables.

In [ ]:
COL_TARGET = "Energikarakter"
COL_CATEGORY = "Bygningskategori"
COL_YEAR_BUILT = "Byggear"
COL_MATERIAL = "Materialvalg"
COL_HEATING = "Oppvarmingskarakter"
COL_MUNICIPALITY = "Knr"
COL_DATE = "Utstedelsesdato"
DWELLING_ID = ["Knr", "Gnr", "Bnr", "Snr", "Fnr", "Bygningsnummer", "BruksEnhetsNummer"]

needed = [COL_TARGET, COL_CATEGORY, COL_YEAR_BUILT, COL_MATERIAL, COL_HEATING, COL_MUNICIPALITY, COL_DATE]
missing = [c for c in needed if c not in raw.columns]
assert not missing, f"Missing columns {missing}. The file has: {list(raw.columns)}"

## 2. Inspect the data

In [ ]:
raw.info()

In [ ]:
# Share of missing values per column
(raw.isna().mean() * 100).round(1).sort_values(ascending=False).to_frame("% missing")

In [ ]:
raw[COL_CATEGORY].value_counts()

In [ ]:
raw[COL_TARGET].value_counts(dropna=False).sort_index()

In [ ]:
raw[COL_YEAR_BUILT].describe()

## 3. Clean the data

- **Homes only:** the register also covers office buildings, schools, etc. We keep the
  residential categories (*Småhus* = detached/small houses, *Boligblokker* = apartment blocks).
- **Valid target:** only grades A–G.
- **Plausible year built:** between 1600 and the certificate's issue year.
- **One certificate per dwelling:** a dwelling can be certified several times. We keep the newest one
  so the same dwelling can't end up in both training and test data.

In [ ]:
RESIDENTIAL = ["Småhus", "Boligblokker"]

df = raw.copy()
n0 = len(df)

df = df[df[COL_CATEGORY].isin(RESIDENTIAL)]
print(f"Homes only:            {len(df):>9,} rows (dropped {n0 - len(df):,})")

df[COL_TARGET] = df[COL_TARGET].astype(str).str.strip().str.upper()
n = len(df); df = df[df[COL_TARGET].isin(list("ABCDEFG"))]
print(f"Valid energikarakter:  {len(df):>9,} rows (dropped {n - len(df):,})")

df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce", format="mixed")
df[COL_YEAR_BUILT] = pd.to_numeric(df[COL_YEAR_BUILT], errors="coerce")
n = len(df)
df = df[df[COL_YEAR_BUILT].isna() | df[COL_YEAR_BUILT].between(1600, df[COL_DATE].dt.year.fillna(2100))]
print(f"Plausible year built:  {len(df):>9,} rows (dropped {n - len(df):,})")

id_cols = [c for c in DWELLING_ID if c in df.columns]
n = len(df)
df = df.sort_values(COL_DATE).drop_duplicates(subset=id_cols, keep="last")
print(f"One per dwelling:      {len(df):>9,} rows (dropped {n - len(df):,})")

if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)
    print(f"Sampled down to {len(df):,} rows")

## 4. Exploratory data analysis (EDA)

In [ ]:
GRADES = list("ABCDEFG")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x=COL_TARGET, order=GRADES, ax=axes[0], color="tab:blue")
axes[0].set_title("Distribution of energikarakter")
(pd.crosstab(df[COL_TARGET], df[COL_CATEGORY], normalize="columns").reindex(GRADES) * 100).plot.bar(ax=axes[1], rot=0)
axes[1].set_title("Energikarakter by building type (% within type)")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x=COL_TARGET, y=COL_YEAR_BUILT, order=GRADES, ax=ax, showfliers=False)
ax.set_title("Year built by energikarakter")

In [ ]:
# Share of each energikarakter per heating grade (rows sum to 100 %)
ct = pd.crosstab(df[COL_HEATING], df[COL_TARGET], normalize="index").reindex(columns=GRADES) * 100
sns.heatmap(ct, annot=True, fmt=".0f", cmap="YlOrRd")
plt.title("Energikarakter (%) per oppvarmingskarakter")

In [ ]:
pd.crosstab(df[COL_MATERIAL].fillna("(missing)"), df[COL_TARGET], normalize="index").reindex(columns=GRADES).round(2)

## 5. Feature engineering

| Feature | Description |
|---|---|
| `bygningskategori` | Småhus / Boligblokker |
| `byggeaar` | Year built |
| `materialvalg` | Construction material (wood, masonry/concrete, …) |
| `fylke` | County, taken from the first two digits of the municipality number. Proxy for climate |
| `attest_aar` | Year the certificate was issued. The calculation rules have changed over time |

**Leakage:** the file also contains `BeregnetLevertEnergiTotaltkWhm2`, the computed energy use per m². The
energikarakter is calculated directly from it, so including it would let the model
"cheat". It's also not something a normal user knows, so we leave it out.

**Not used: `oppvarmingskarakter`.** The heating grade is a strong predictor (see the heatmap in the EDA),
but it's printed on the energy certificate itself. Someone who doesn't have a certificate yet, which is
who the app is for, won't know it. It's also never missing in the data, so the model couldn't learn
what to do when a user leaves it blank. We trade some accuracy for a model people can actually use.

**Target:** A=1, B=2, …, G=7.

In [ ]:
# The same material is spelled in several ways in the data
MATERIAL_FIX = {"MurTeglstein": "Mur/tegl", "Stal": "Stål"}


def make_features(d: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "bygningskategori": d[COL_CATEGORY].astype(str),
        "byggeaar": pd.to_numeric(d[COL_YEAR_BUILT], errors="coerce"),
        "materialvalg": d[COL_MATERIAL].astype("string").str.strip().replace(MATERIAL_FIX),
        # Knr is blank in a few rows, so convert with errors="coerce" (blank -> missing)
        "fylke": pd.to_numeric(d[COL_MUNICIPALITY], errors="coerce").astype("Int64").astype("string").str.zfill(4).str[:2],
        "attest_aar": pd.to_datetime(d[COL_DATE], errors="coerce").dt.year,
    }, index=d.index)


GRADE_TO_NUM = {g: i + 1 for i, g in enumerate(GRADES)}
NUM_TO_GRADE = {v: k for k, v in GRADE_TO_NUM.items()}

X = make_features(df)
y = df[COL_TARGET].map(GRADE_TO_NUM).astype(float)

NUMERIC = ["byggeaar", "attest_aar"]
CATEGORICAL = ["bygningskategori", "materialvalg", "fylke"]
X = X[NUMERIC + CATEGORICAL]
# The pipeline's SimpleImputer expects np.nan, not pd.NA
X[CATEGORICAL] = X[CATEGORICAL].astype(object).where(X[CATEGORICAL].notna(), np.nan)
X.head()

## 6. Train/test split and preprocessing

We hold out 20 % as a **test set** that we only touch at the very end. The split is stratified on
energikarakter so both sets get the same grade distribution.

The preprocessing goes inside a `Pipeline`, so exactly the same transformations are applied in
training and prediction, and cross-validation doesn't leak information between folds.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="ukjent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50, sparse_output=False)),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC),
    ("cat", categorical_pipe, CATEGORICAL),
])

## 7. Compare models with cross-validation

We compare:
- **Baseline:** always predicts the median. Every real model has to beat this.
- **Ridge regression:** linear model.
- **Random forest**
- **Histogram gradient boosting:** usually strong on tabular data, and fast on large datasets.

Metrics:
- **MAE**: average error in number of grades (0.5 = half a grade off on average)
- **Exact accuracy**: share where the rounded prediction hits the right grade exactly
- **±1 accuracy**: share that is at most one grade off

In [ ]:
def to_grade(pred):
    return np.clip(np.rint(pred), 1, 7)


def exact_accuracy(est, X, y):
    return np.mean(to_grade(est.predict(X)) == y)


def within_one(est, X, y):
    return np.mean(np.abs(to_grade(est.predict(X)) - y) <= 1)


scoring = {
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "exact": exact_accuracy,
    "within_1": within_one,
}

models = {
    "Baseline (median)": DummyRegressor(strategy="median"),
    "Ridge": Ridge(alpha=1.0),
    "Random forest": RandomForestRegressor(n_estimators=150, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient boosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

# CV on at most 200k rows so it doesn't take forever on the full register
cv_n = min(len(X_train), 200_000)
X_cv, y_cv = X_train.sample(cv_n, random_state=RANDOM_STATE), y_train.sample(cv_n, random_state=RANDOM_STATE)
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    s = cross_validate(pipe, X_cv, y_cv, cv=cv, scoring=scoring, n_jobs=1)
    results[name] = {
        "MAE": -s["test_MAE"].mean(),
        "R²": s["test_R2"].mean(),
        "Exact accuracy": s["test_exact"].mean(),
        "±1 accuracy": s["test_within_1"].mean(),
        "Time (s)": s["fit_time"].mean(),
    }
    print(f"{name:20s} MAE={results[name]['MAE']:.3f}")

pd.DataFrame(results).T.round(3)

## 8. Hyperparameter tuning

We tune gradient boosting, which usually comes out best, with a random search over a few
hyperparameters. If another model won above, swap it in here.

In [ ]:
search = RandomizedSearchCV(
    Pipeline([("prep", preprocess), ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
    param_distributions={
        "model__learning_rate": [0.03, 0.05, 0.1, 0.2],
        "model__max_leaf_nodes": [15, 31, 63, 127],
        "model__min_samples_leaf": [20, 50, 100, 200],
        "model__l2_regularization": [0.0, 0.1, 1.0],
        "model__max_iter": [200, 400, 800],
    },
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
search.fit(X_cv, y_cv)
print("Best parameters:", search.best_params_)
print(f"CV MAE: {-search.best_score_:.3f}")

## 9. Evaluate on the test set

Retrain the best model on all the training data, then evaluate **once** on the test set.

In [ ]:
best_model = search.best_estimator_
best_model.fit(X_train, y_train)

pred = best_model.predict(X_test)
pred_grade = to_grade(pred)

print(f"MAE (raw):          {mean_absolute_error(y_test, pred):.3f} grades")
print(f"RMSE:               {root_mean_squared_error(y_test, pred):.3f}")
print(f"R²:                 {r2_score(y_test, pred):.3f}")
print(f"Exact accuracy:     {np.mean(pred_grade == y_test):.1%}")
print(f"±1 grade accuracy:  {np.mean(np.abs(pred_grade - y_test) <= 1):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test.map(NUM_TO_GRADE), pd.Series(pred_grade).map(NUM_TO_GRADE),
    labels=GRADES, normalize="true", values_format=".0%", cmap="Blues", ax=ax, colorbar=False,
)
ax.set_title("Confusion matrix (share per true grade)")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")

In [ ]:
# Distribution of errors: how many grades off?
pd.Series(pred_grade - y_test.values).value_counts(normalize=True).sort_index().mul(100).round(1).to_frame("% of test set")

## 10. Which features matter?

**Permutation importance:** shuffle one feature at a time and measure how much worse the model gets.
A big increase in error means the feature is important.

In [ ]:
n_pi = min(len(X_test), 20_000)
X_pi, y_pi = X_test.iloc[:n_pi], y_test.iloc[:n_pi]
pi = permutation_importance(best_model, X_pi, y_pi, scoring="neg_mean_absolute_error",
                            n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_pi.columns).sort_values()
imp.plot.barh(figsize=(7, 4), title="Permutation importance (increase in MAE)")

## 11. Save the model and try it out

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(best_model, MODEL_DIR / "energikarakter_model.joblib")


def predict_energikarakter(bygningskategori, byggeaar, materialvalg=None, kommunenummer=None, attest_aar=2025):
    """Predicts the energikarakter (A–G) of a single home."""
    row = pd.DataFrame([{
        "byggeaar": byggeaar,
        "attest_aar": attest_aar,
        "bygningskategori": bygningskategori,
        "materialvalg": materialvalg if materialvalg is not None else np.nan,
        "fylke": str(kommunenummer).zfill(4)[:2] if kommunenummer else np.nan,
    }])[NUMERIC + CATEGORICAL]
    value = best_model.predict(row)[0]
    return NUM_TO_GRADE[int(to_grade(value))], round(float(value), 2)


examples = [
    dict(bygningskategori="Småhus", byggeaar=1965, materialvalg="Tre", kommunenummer=301),
    dict(bygningskategori="Småhus", byggeaar=2018, materialvalg="Tre", kommunenummer=5001),
    dict(bygningskategori="Boligblokker", byggeaar=1985, materialvalg="Betong", kommunenummer=4601),
]
for e in examples:
    grade, value = predict_energikarakter(**e)
    print(f"{e} -> {grade} ({value})")

## 12. Evaluation

Section 9 gave the overall numbers. Here we look closer at the final model on the test set: how much
better is it than the baseline, which grades does it get right, and for which homes does it miss?

**Bias** below is the average of *predicted − true* (in grades). Positive means the model predicts
a worse grade than the real one, negative means it's too optimistic.

In [ ]:
baseline = DummyRegressor(strategy="median").fit(X_train, y_train)


def summarize(p):
    g = to_grade(p)
    return {
        "MAE": mean_absolute_error(y_test, p),
        "R²": r2_score(y_test, p),
        "Exact accuracy": np.mean(g == y_test),
        "±1 accuracy": np.mean(np.abs(g - y_test) <= 1),
    }


pd.DataFrame({"Baseline (median)": summarize(baseline.predict(X_test)), "Final model": summarize(pred)}).T.round(3)

### Per grade

In [ ]:
ev = X_test.assign(
    true=y_test.map(NUM_TO_GRADE),
    pred=pd.Series(pred_grade, index=X_test.index).map(NUM_TO_GRADE),
    error=np.abs(pred_grade - y_test),
    bias=pred - y_test,
)


def error_table(by):
    return ev.groupby(by, observed=True).agg(
        n=("error", "size"),
        exact=("error", lambda e: (e == 0).mean()),
        within_1=("error", lambda e: (e <= 1).mean()),
        bias=("bias", "mean"),
    ).round(2)


error_table("true").reindex(GRADES)

In [ ]:
# Does the model predict the same spread of grades as reality?
dist = pd.DataFrame({
    "True": ev["true"].value_counts(normalize=True),
    "Predicted": ev["pred"].value_counts(normalize=True),
}).reindex(GRADES).fillna(0) * 100
dist.plot.bar(rot=0, figsize=(8, 4), ylabel="% of test set", title="True vs predicted grade distribution")

### Where does the model miss?

In [ ]:
error_table("bygningskategori")

In [ ]:
periods = pd.cut(ev["byggeaar"], [1600, 1950, 1970, 1990, 2010, 2026],
                 labels=["før 1950", "1950–69", "1970–89", "1990–2009", "2010–"], right=False)
error_table(periods.rename("byggeperiode"))

In [ ]:
# Material, including homes where it's missing in the data
error_table(ev["materialvalg"].fillna("(mangler)"))

## 13. Conclusion

*Numbers from the last full run (1.3 million homes, 263 475 in the test set).*

**How good is the model?**
- The final model (tuned gradient boosting) predicts the exact grade for **54 %** of homes, and is at most one grade off for **95 %**.
  The baseline that always guesses the median manages 15 % and 50 %.
- On average it misses by **0.58 grades** (MAE), against 1.54 for the baseline, and explains about 82 % of the variation (R² = 0.82).
- Random forest scored about the same in cross-validation, so the choice between the two matters little. The linear model (Ridge) was clearly worse,
  which tells us the relationships are not linear. Year built, for example, doesn't affect the grade in a straight line.

**Where does it miss?**
- **The extreme grades.** Homes that really are A are almost always predicted as B or worse (12 % exact, bias +0.86), and G homes
  are often predicted as F (bias −0.57). This is typical for regression: when unsure, it pulls towards the middle.
  The features can't tell an A home apart from a well-built B home, since the difference is things like heat pumps and insulation.
- **Homes built 1950–1989** are hardest (46–49 % exact). These have often been renovated to very different degrees, which our data
  doesn't capture. Homes built before 1950 are easiest (68 %), because most of them have poor grades.
- **No systematic skew** between building types, periods or materials: the bias is about 0 in every group, so the model isn't
  consistently too optimistic or pessimistic for any kind of home.
- **Missing material** gives lower accuracy (47 %). That's one reason the web app requires the user to choose a material.

**Choices that cost accuracy on purpose**
- Leaving out `oppvarmingskarakter` (section 5) lowered exact accuracy from about 60 % to 54 %. The model is less precise,
  but it only asks for things a homeowner actually knows.
- `BeregnetLevertEnergiTotaltkWhm2` was left out because it would let the model cheat (leakage).
